# CRIVO — Escola Conceitual REAL (GPU)

Este notebook altera **os pesos reais do Transformer CRIVO 17,43M**. A intervenção nova é aprendizado de **invariância semântica**: formulações diferentes do mesmo fato devem ficar próximas internamente.

Base: checkpoint V7. Arquitetura e tokenizer permanecem CRIVO. Embeddings e blocos 0–3 ficam congelados; blocos 4–7 + norm treinam. Nenhum modelo externo/API. O resultado permanece `ativo_no_chat=false`.


In [ ]:
!nvidia-smi
!pip -q install 'tokenizers>=0.20,<1'
import torch, pathlib, random, json, hashlib, shutil, zipfile, time, sys
assert torch.cuda.is_available(), 'Ative GPU no Colab.'
DEVICE='cuda'; random.seed(20261011); torch.manual_seed(20261011)
from google.colab import drive
drive.mount('/content/drive')
DRIVE=pathlib.Path('/content/drive/MyDrive/CRIVO/checkpoints'); DRIVE.mkdir(parents=True,exist_ok=True)
ZIP=DRIVE/'crivo_v7_checkpoint.zip'
assert ZIP.exists(), f'Não encontrei {ZIP}'
print('✅ GPU:',torch.cuda.get_device_name(0)); print('✅ checkpoint:',ZIP)


In [ ]:
%cd /content
!rm -rf CRIVO crivo_v7_base
!git clone -q https://github.com/HROSONE/CRIVO.git
BASE=pathlib.Path('/content/crivo_v7_base'); BASE.mkdir()
with zipfile.ZipFile(ZIP) as z: z.extractall(BASE)
sys.path.insert(0,'/content/CRIVO')
from linguagem_profunda import Configuracao,LinguagemProfunda
from tokenizers import Tokenizer
estado=torch.load(BASE/'pesos.pt',map_location='cpu',weights_only=True)
CFG={'vocabulario':8192,'dimensao':384,'camadas':8,'cabecas':8,'contexto':256,'dropout':0.1}
assert estado['config']==CFG
modelo=LinguagemProfunda(Configuracao(**CFG)); modelo.load_state_dict(estado['modelo'],strict=True); modelo=modelo.to(DEVICE)
tok=Tokenizer.from_file(str(BASE/'tokenizer.json')); tok.encode_special_tokens=True
assert sum(p.numel() for p in modelo.parameters())==17428224
PAD=tok.token_to_id('<pad>'); DOC=tok.token_to_id('<documento>'); FIM=tok.token_to_id('<fim>')
print('✅ CRIVO real carregado: 17.428.224 parâmetros | passo origem:',estado.get('passo'))


## Base conceitual
Cada conceito aparece como categoria, função, propriedade e relação. O mesmo fato ganha várias superfícies linguísticas. Dois templates extras ficam fora do treino para avaliação.


In [ ]:
C=[
('copo','recipiente',['conter líquidos','permitir beber'],['pode ser de vidro, plástico, metal ou barro','água pode ficar dentro de um copo']),
('garrafa','recipiente',['armazenar líquidos','transportar bebidas'],['pode ter tampa','água pode ser armazenada em uma garrafa']),
('cadeira','móvel',['servir de assento','sustentar uma pessoa sentada'],['pode ser de madeira, metal ou plástico','uma pessoa pode sentar em uma cadeira']),
('martelo','ferramenta',['golpear objetos','cravar pregos'],['é segurado pela mão','pode ser usado para cravar um prego']),
('telescópio','instrumento óptico',['coletar luz','observar objetos distantes'],['usa elementos ópticos','pode ser apontado para o céu']),
('microscópio','instrumento óptico',['ampliar estruturas pequenas','observar detalhes microscópicos'],['uma amostra pode ser observada em um microscópio']),
('termômetro','instrumento de medição',['medir temperatura'],['produz uma leitura de temperatura']),
('computador','máquina eletrônica',['processar informações','executar programas'],['executa instruções codificadas']),
('água','substância',['hidratar organismos','atuar como solvente'],['é líquida em condições comuns','ocupa o recipiente que a contém']),
('planeta','corpo celeste',['orbitar uma estrela'],['tem massa e gravidade','a Terra é um planeta']),
('hipótese','proposição provisória',['representar uma possibilidade a testar'],['não equivale automaticamente a fato confirmado','pode ser avaliada por evidências']),
('contexto','informação circundante',['ajudar a interpretar palavras e referências'],['pode mudar o significado de uma expressão','ajuda a resolver referências']),
('memória','processo de armazenamento e recuperação',['reter informação','recuperar informação anterior'],['depende de informação registrada anteriormente']),
('correção','atualização de informação',['substituir informação anterior errada'],['muda o estado vigente da conversa']),
('pergunta','ato de linguagem',['solicitar informação','pedir esclarecimento'],['expressa uma lacuna ou pedido']),
('resposta','ato de linguagem',['fornecer informação solicitada'],['deve considerar o contexto'])]
T=['{n} é um tipo de {v}.','{n} pertence à categoria ou relação {v}.','Sobre {n}, vale que {v}.','Uma forma de entender {n}: {v}.']
H=['Mesmo usando outras palavras, sobre {n} continua verdadeiro que {v}.','Reformulando a ideia: {n} se relaciona a {v}.']
fatos=[]
for n,cat,fun,prop in C:
    fatos.append((n,cat))
    fatos += [(n,x) for x in fun+prop]
treino=[]; pares=[]; hold=[]
for n,v in fatos:
    formas=[t.format(n=n,v=v) for t in T]
    treino+=formas; pares += [(formas[0],formas[2]),(formas[1],formas[3])]
    hold += [h.format(n=n,v=v) for h in H]
base=list(treino)
for p in ['Em termos simples, ','Considere o seguinte: ','No uso comum, ','Como relação conceitual, ']: treino += [p+s[0].lower()+s[1:] for s in base]
random.shuffle(treino); random.shuffle(pares)
print('Fatos:',len(fatos),'| frases treino:',len(treino),'| pares equivalentes:',len(pares),'| holdout:',len(hold))


In [ ]:
def enc(s,doc=True):
    x=tok.encode(s,add_special_tokens=False).ids
    return ([DOC]+x+[FIM] if doc else x)[:256]
def batch_lm(bs=16):
    ss=random.sample(treino,bs); q=[enc(s) for s in ss]; L=max(map(len,q))-1
    x=torch.full((bs,L),PAD,dtype=torch.long,device=DEVICE); y=torch.full_like(x,-100)
    for i,s in enumerate(q): x[i,:len(s)-1]=torch.tensor(s[:-1],device=DEVICE); y[i,:len(s)-1]=torch.tensor(s[1:],device=DEVICE)
    return x,y
def hidden(ids):
    t=ids.shape[1]; h=modelo.embedding(ids)+modelo.posicao(torch.arange(t,device=DEVICE))
    for b in modelo.blocos: h=b(h)
    return modelo.norm(h)
def reprs(frases):
    q=[enc(s,False) for s in frases]; L=max(map(len,q)); x=torch.full((len(q),L),PAD,dtype=torch.long,device=DEVICE); m=torch.zeros((len(q),L),dtype=torch.bool,device=DEVICE)
    for i,s in enumerate(q): x[i,:len(s)]=torch.tensor(s,device=DEVICE); m[i,:len(s)]=1
    h=hidden(x); z=(h*m[:,:,None]).sum(1)/m.sum(1,keepdim=True).clamp_min(1)
    return torch.nn.functional.normalize(z,dim=-1)
def inv_loss(pb,temp=.08):
    a=[x for x,_ in pb]; b=[y for _,y in pb]; za=reprs(a); zb=reprs(b); logits=za@zb.T/temp; lab=torch.arange(len(pb),device=DEVICE)
    return (torch.nn.functional.cross_entropy(logits,lab)+torch.nn.functional.cross_entropy(logits.T,lab))/2
@torch.no_grad()
def ce_lista(frases):
    modelo.eval(); total=0.; nt=0
    for s in frases:
        q=enc(s); x=torch.tensor([q[:-1]],device=DEVICE); y=torch.tensor([q[1:]],device=DEVICE); log,_=modelo(x)
        total+=float(torch.nn.functional.cross_entropy(log.reshape(-1,8192),y.reshape(-1),reduction='sum')); nt+=y.numel()
    return total/nt
@torch.no_grad()
def retrieval(ps):
    modelo.eval(); ps=ps[:96]; za=reprs([a for a,b in ps]); zb=reprs([b for a,b in ps]); sim=za@zb.T; lab=torch.arange(len(ps),device=DEVICE)
    return float((sim.argmax(1)==lab).float().mean())
pares_hold=[]
for n,v in fatos: pares_hold.append((H[0].format(n=n,v=v),H[1].format(n=n,v=v)))
PRES=['A Lua orbita a Terra e a Terra orbita o Sol.','O coração bombeia sangue através dos vasos do corpo.','Uma biblioteca organiza livros para que possam ser encontrados novamente.','Uma conclusão deve respeitar as evidências disponíveis.','A pressão atmosférica varia com altitude e condições meteorológicas.']
BASELINE={'holdout_ce':ce_lista(hold),'retrieval_top1':retrieval(pares_hold),'preservacao_ce':ce_lista(PRES)}
print('ANTES:',json.dumps(BASELINE,indent=2,ensure_ascii=False))


## TREINO REAL
Esta célula modifica o CRIVO. Salva recuperação parcial no Drive a cada 250 passos e o checkpoint final ao terminar.


In [ ]:
for p in modelo.parameters(): p.requires_grad=False
for b in modelo.blocos[4:]:
    for p in b.parameters(): p.requires_grad=True
for p in modelo.norm.parameters(): p.requires_grad=True
pars=[p for p in modelo.parameters() if p.requires_grad]
STEPS=1200; LR=1e-5; LAMBDA_INV=.35; opt=torch.optim.AdamW(pars,lr=LR,weight_decay=.01); scaler=torch.amp.GradScaler('cuda')
OUTROOT=DRIVE/'escola_conceitual_real'; OUTROOT.mkdir(exist_ok=True); PARCIAL=OUTROOT/'pesos_parcial.pt'; LOG=[]; rng=random.Random(2026101102); ini=time.time()
for step in range(1,STEPS+1):
    modelo.train(); [b.eval() for b in modelo.blocos[:4]]; x,y=batch_lm(16); pb=rng.sample(pares,16); opt.zero_grad(set_to_none=True)
    with torch.autocast(device_type='cuda',dtype=torch.float16):
        log,_=modelo(x); lm=torch.nn.functional.cross_entropy(log.reshape(-1,8192),y.reshape(-1),ignore_index=-100); inv=inv_loss(pb); loss=lm+LAMBDA_INV*inv
    scaler.scale(loss).backward(); scaler.unscale_(opt); grad=float(torch.nn.utils.clip_grad_norm_(pars,1.0)); scaler.step(opt); scaler.update()
    if step%50==0 or step==1:
        r={'step':step,'loss':float(loss.detach()),'lm':float(lm.detach()),'inv':float(inv.detach()),'grad':grad,'seg':round(time.time()-ini,1)}; LOG.append(r); print(r)
    if step%250==0:
        e={'versao':estado['versao'],'modelo':modelo.state_dict(),'config':estado['config'],'passo':int(estado.get('passo',0))+step,'execucao':{**estado.get('execucao',{}),'escola_conceitual':{'step':step,'ativo_no_chat':False}}}; torch.save(e,PARCIAL); print('💾 parcial:',PARCIAL)
print('✅ treino terminou em',round(time.time()-ini,1),'s')


In [ ]:
FINAL={'holdout_ce':ce_lista(hold),'retrieval_top1':retrieval(pares_hold),'preservacao_ce':ce_lista(PRES)}
REL={'holdout_ce_ratio':FINAL['holdout_ce']/BASELINE['holdout_ce'],'retrieval_gain':FINAL['retrieval_top1']-BASELINE['retrieval_top1'],'preservacao_ce_ratio':FINAL['preservacao_ce']/BASELINE['preservacao_ce']}
stamp=time.strftime('%Y%m%d_%H%M%S'); OUT=OUTROOT/f'crivo_conceitual_{stamp}'; OUT.mkdir(parents=True)
novo={'versao':estado['versao'],'modelo':modelo.state_dict(),'config':estado['config'],'passo':int(estado.get('passo',0))+STEPS,'execucao':{**estado.get('execucao',{}),'escola_conceitual':{'base':'V7','intervencao_unica':'invariancia_semantica_contrastiva','steps':STEPS,'lr':LR,'lambda':LAMBDA_INV,'blocos_treinaveis':[4,5,6,7],'ativo_no_chat':False}}}
torch.save(novo,OUT/'pesos.pt'); shutil.copy2(BASE/'tokenizer.json',OUT/'tokenizer.json'); shutil.copy2(BASE/'ponte_binding.pt',OUT/'ponte_binding_v7_nao_revalidada.pt')
meta={'ativo_no_chat':False,'promovido':False,'ponte_v7_requer_revalidacao':True,'baseline':BASELINE,'final':FINAL,'relativo':REL,'log':LOG}; (OUT/'metricas.json').write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf-8')
h=hashlib.sha256((OUT/'pesos.pt').read_bytes()).hexdigest()
print('ANTES:',BASELINE); print('DEPOIS:',FINAL); print('MUDANÇA:',REL); print('✅ CHECKPOINT FINAL:',OUT); print('SHA256:',h); print('ativo_no_chat=false')
